# 팀 프로젝트2 — 14조: 다음 학년도 중도탈락 고위험 학과는 어디인가
학업지속 지원 대상 학과 선정을 위한 대학알리미 학과별 중도탈락 패널(2023~2026) 분석 및 예측

- 팀: 정승엽, 박유진, 신효주 · 담당교수 박주원
- Colab 정본 링크: `[팀 입력]`
- 실행 방법: CSV를 올린 뒤 **런타임 → 다시 시작 및 모두 실행**. 처음부터 재실행할 수 있어야 한다(불가하면 −5점).
- 구성: 안내문 5단계 **① 문제 정의 → ② 수집 → ③ 정제 → ④ 분석 → ⑤ 발표**, 그리고 부록(재현성 확인). 기준 문서는 `docs/기준_정리.md`.
- **출처 규칙**: 수치를 출력하는 셀에는 맨 위 주석에 셀 ID(예: `s3-06`)를 둔다. 보고서·발표에 옮길 때 `(출처: 노트북 §③ 셀 s3-06)` 형식으로 붙인다. 출처 없는 수치는 건당 −2점.
- **고정 규칙**: 집단별 탈락률은 `hypotheses-locked-v3` 태그(2026-11-11) 이후에만 계산한다. 분류는 데이터 컬럼만 쓰고 수작업 매핑은 만들지 않는다.

## 0. 환경·경로

In [1]:
# s0-01 환경·경로·스위치
import os, glob, platform, datetime
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

# 원자료 경로 후보: Colab 업로드(/content) → Drive → 로컬(notebook/ 기준 ../data/raw)
FNAME = '교육부_대학알리미_중도탈락 학생 현황 _대학_학과별_20260831.csv'
CANDIDATES = [f'/content/{FNAME}', f'/content/drive/MyDrive/{FNAME}', f'../data/raw/{FNAME}', f'data/raw/{FNAME}']
CANDIDATES += sorted(glob.glob('/content/*중도탈락*.csv')) + sorted(glob.glob('../data/raw/*중도탈락*.csv'))
CSV_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError('원자료 CSV를 찾지 못했습니다. Colab이면 왼쪽 파일 아이콘으로 CSV를 /content에 올려 주세요.')
if CSV_PATH.startswith('../'):
    PROC_DIR = '../data/processed'
elif CSV_PATH.startswith('data/'):
    PROC_DIR = 'data/processed'
else:
    PROC_DIR = '/content/processed'
FIG_DIR = 'figures'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# 고정값·되돌림 스위치 (logs/정제로그.md 표 6, 00_문제정의_가설.md와 이름 일치)
RANDOM_STATE = 42
SCOPE = 'general_day'            # 'general_day'(일반 대학 주간, 사전 고정) | 'all'(전체 표본, 민감도 ②)
MIN_ENROLL = 30                  # 직전연도 최소 재적(명), 민감도 10·50
MIN_ENROLL_SENS = [10, 30, 50]
HIGH_PCT = 75                    # 고위험 컷 백분위(상위 25%), 민감도 80·70
RATE_SOURCE = 'published'        # 공시 비율 사용(팀 결정 2026-11-11)
HUFS_CODE = 192                  # 한국외국어대학교 (사이버한국외국어대학교 332와 구분)
B_SUBFIELDS = ['기타유럽어・문학', '기타아시아어・문학']
B_SUBFIELDS_SENS = B_SUBFIELDS + ['국제지역학']   # 민감도 ①
TRAIN_T, TEST_T = [2024, 2025], 2026

print('실행 시각:', datetime.datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)
print('CSV_PATH =', CSV_PATH)
print('PROC_DIR =', PROC_DIR, '| FIG_DIR =', FIG_DIR)

실행 시각: 2026-11-11 23:05
Python 3.13.14 | pandas 3.0.6 | numpy 2.5.3
CSV_PATH = ../data/raw/교육부_대학알리미_중도탈락 학생 현황 _대학_학과별_20260831.csv
PROC_DIR = ../data/processed | FIG_DIR = figures


## ① 문제 정의 (15점)
- **무엇을**: 의사결정 질문, 가설 A·B, 판정 규칙, 반대 결과 처리 규칙을 인용한다(코드 없음).
- **왜**: 결과를 보기 전에 고정했다는 것을 보여 주려고. 이 셀은 수정하지 않는다.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: git 태그 `hypotheses-locked`(v1, 2026-11-08), `hypotheses-locked-v2`(2026-11-11), `00_문제정의_가설.md` 변경 이력

> `00_문제정의_가설.md` v2 인용 (요약). 상세 판정 규칙은 원문을 따른다.

- **누가**: 한국외국어대학교 학사종합지원센터 (보조: 입학처)
- **무엇을**: 다음 학년도 학업지속 지원(상담·학습지원·복학 안내)을 우선 배치할 학과
- **질문 ①**: 전국 일반 대학 학과 중 어떤 설립유형·지역에서 고위험(상위 25%)이 많은가
- **질문 ②**: 그중 한국외대에서는 어떤 학과군이 고위험인가
- **가설 A (전국 구조)**: 사립·비수도권 학과는 고위험 가능성이 높다.
  - 판정: 대계열·전년도 재적을 통제한 로지스틱 회귀에서 두 오즈비가 모두 1보다 크고 95% CI가 1을 포함하지 않으면 맞음. 하나만이면 부분 맞음, 둘 다 아니면 기각.
- **가설 B (한국외대)**: 비주류 외국어 학과군(소계열 기타유럽어・문학·기타아시아어・문학)은 교내 다른 학과보다 탈락률이 높다.
  - 판정: 2024~2026 합산 비율 차이의 부트스트랩 95% 구간 하한이 0보다 크면 맞음. 아니면 기각하고 '모집(입학처)'으로 재정의.
- **예측 절**: A·B 안에서 모델과 기준선 2개를 같은 k의 정밀도로 비교한다. 판정 임계값 없이 정직하게 보고한다.
- **반대 결과**: 그대로 받아들이고 결론에 맞았다/부분 맞았다/틀렸다를 명시한다.

## ② 수집
- **무엇을**: 원본을 읽고 출처·URL·수집일·이용 조건, 행·열 수, 측정 척도를 출력한다.
- **왜**: 데이터가 공개·실제 자료이고 계획서의 규모와 같다는 것을 확인하려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 기대 행·열 수 assert, `docs/수집_기록표.md`와 출력 대조

In [2]:
# s2-01 출처 표기
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §② 셀 s2-01)'를 붙인다.
SOURCE_INFO = {
    '데이터명': '교육부_대학알리미_중도탈락 학생 현황_대학_학과별',
    '제공처': '공공데이터포털 15164570 (https://www.data.go.kr/data/15164570/fileData.do)',
    '제공기관': '교육부 교육데이터정책과 (원 출처: 대학알리미 대학 정보공시)',
    '이용허락': '공공누리 제1유형 (출처 표시)',
    '포털 수정일 / 파일명 날짜': '2026-09-10 / 20260831',
    '다운로드일': '2026-11-08',
}
for k, v in SOURCE_INFO.items():
    print(f'{k:>16}: {v}')

            데이터명: 교육부_대학알리미_중도탈락 학생 현황_대학_학과별
             제공처: 공공데이터포털 15164570 (https://www.data.go.kr/data/15164570/fileData.do)
            제공기관: 교육부 교육데이터정책과 (원 출처: 대학알리미 대학 정보공시)
            이용허락: 공공누리 제1유형 (출처 표시)
 포털 수정일 / 파일명 날짜: 2026-09-10 / 20260831
           다운로드일: 2026-11-08


In [3]:
# s2-02 원본 읽기
# [출처 규칙] '(출처: 노트북 §② 셀 s2-02)'
# 원본 df_raw는 이후 수정하지 않는다. 정제는 사본 df에서 한다.
df_raw = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print('행 × 열 =', df_raw.shape)
assert df_raw.shape == (56340, 32), '계획서 기준값(56,340×32)과 다름 → 원자료 버전을 확인'

행 × 열 = (56340, 32)


In [4]:
# s2-03 공시연도별 행 수
# [출처 규칙] '(출처: 노트북 §② 셀 s2-03)'
year_n = df_raw['공시연도'].value_counts().sort_index()
print(year_n.to_string())
assert year_n.to_dict() == {2023: 14350, 2024: 13628, 2025: 13928, 2026: 14434}
print('\n공시연도 × 기준연도 (대부분 기준연도 = 공시연도 − 1, 일부 같은 해)')
print(pd.crosstab(df_raw['공시연도'], df_raw['기준연도']))

공시연도
2023    14350
2024    13628
2025    13928
2026    14434

공시연도 × 기준연도 (대부분 기준연도 = 공시연도 − 1, 일부 같은 해)
기준연도   2022   2023   2024   2025  2026
공시연도                                  
2023  14178    172      0      0     0
2024      0  13484    144      0     0
2025      0      0  13827    101     0
2026      0      0      0  14332   102


In [5]:
# s2-04 컬럼 개요와 측정 척도
# [출처 규칙] '(출처: 노트북 §② 셀 s2-04)'
SCALE = {'공시연도': '등간 또는 명목', '기준연도': '등간 또는 명목', '재적학생': '비율 척도(명)', '계': '비율 척도(명)',
         '중도탈락학생비율': '비율 척도(%)'}
REASONS = ['미등록', '미복학', '자퇴', '학사경고', '학생활동', '유급제적', '재학연한초과', '기타']
SCALE.update({c: '비율 척도(명)' for c in REASONS})
overview = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    '고유값 수': df_raw.nunique(),
    '결측 수': df_raw.isna().sum(),
    '예시값': [df_raw[c].dropna().iloc[0] for c in df_raw.columns],
    '측정 척도': [SCALE.get(c, '명목') for c in df_raw.columns],
})
overview

,dtype,고유값 수,결측 수,예시값,측정 척도
공시연도,int64,4,0,2026,등간 또는 명목
기준연도,int64,5,0,2025,등간 또는 명목
학교코드,int64,250,0,2748,명목
학교명,str,243,0,가야대학교(김해),명목
본분교명,str,5,0,본교,명목
대학구분,str,1,0,대학,명목
학교종류,str,7,0,대학교,명목
설립구분,str,6,0,사립,명목
설립유형,str,2,0,사립,명목
소재지유형,str,2,0,비수도권,명목


## ③ 정제 (`logs/정제로그.md` 표 2 순서)
- **무엇을**: 품질 6대 기준으로 사전 진단하고, 범위를 확정하고, 단계별로 이슈를 표시(flag)·추적한다.
- **왜**: 어떤 이슈가 분석 범위·표본에 영향을 주는지 먼저 보고 처리 여부를 정하려고. 행은 지우지 않고 표시 후 표본에서 제외한다.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 기대값 assert, 단계별 추적표(정제로그 표 5), 처리·건수·근거·영향 로그(표 4), 0행 필터 검사

In [6]:
# s3-00 단계별 추적 도구
# [기준] 행 삭제 금지(표시 후 표본에서 제외), y(비율) 보정 금지, 건수는 코드 계산값만, 0행 필터는 멈춤(assert)
KEY = ['공시연도', '학교코드', '학과코드']
LOG = []          # 정제 처리 로그(정제로그 표 4): 처리·건수·근거·영향
TRACK = {}        # 단계별 추적표(정제로그 표 5)

def log(no, 처리, 건수, 근거, 영향):
    LOG.append({'#': no, '처리': 처리, '건수': 건수, '근거': 근거, '영향': 영향})

def make_flags(d):
    """이슈별 불리언 표. 값은 바꾸지 않는다."""
    f = pd.DataFrame(index=d.index)
    enr, cnt, rate = d['재적학생'], d['계'], d['중도탈락학생비율']
    recomputed = cnt / enr.where(enr > 0) * 100
    f['재적 = 0'] = enr.eq(0)
    f['재적 = 0 이면서 계 > 0'] = enr.eq(0) & cnt.gt(0)
    f['재적 = 0 인데 비율 기록됨(0%)'] = enr.eq(0) & rate.notna()
    f['계 > 재적 (재적 > 0)'] = enr.gt(0) & cnt.gt(enr)
    f['재계산 불일치 (> 0.01%p)'] = (rate - recomputed).abs().gt(0.01)
    f['사유합 ≠ 계'] = cnt.notna() & d[REASONS].sum(axis=1, min_count=1).ne(cnt)
    f['비율 결측'] = rate.isna()
    f['└ 계 결측'] = cnt.isna()
    f['└ 재적 0 (계 있음)'] = rate.isna() & cnt.notna() & enr.eq(0)
    f['폐교'] = d['학교상태'].eq('폐교')
    f['신설'] = d['학교상태'].eq('신설')
    f['소계열 N.C.E.'] = d['표준분류_소계열'].eq('N.C.E.')
    f['중계열 N.C.E'] = d['표준분류_중계열'].eq('N.C.E')
    return f

def rate_valid(d):
    """y로 쓸 수 있는 비율: 재적 > 0, 계·비율 존재 (공시값 그대로)"""
    return d['재적학생'].gt(0) & d['계'].notna() & d['중도탈락학생비율'].notna()

def track(stage, d):
    t = make_flags(d).sum().astype(int)
    TRACK[stage] = pd.concat([pd.Series({'(참고) 행 수': len(d)}), t])
    assert len(d) > 0, f'{stage}: 0행 — 조건을 의심할 것'
    return TRACK[stage]

df = df_raw.copy()     # 정제용 사본
track('① 원본 전체', df);
print('추적 도구 준비 완료. 이슈 항목 수:', len(TRACK['① 원본 전체']) - 1)

추적 도구 준비 완료. 이슈 항목 수: 13


### ③-0 사전 진단 (품질 6대 기준, 읽기 전용)
> 기대값은 계획서 기준값이며 assert로 대조한다. 다르면 셀이 멈춘다.

In [7]:
# s3-01 [완전성] 결측
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-01)'
f0 = make_flags(df)
n_rate_na, n_cnt_na, n_z = f0['비율 결측'].sum(), f0['└ 계 결측'].sum(), f0['└ 재적 0 (계 있음)'].sum()
print(f'비율 결측 {n_rate_na:,} = 계 결측 {n_cnt_na:,} + 재적 0(계 있음) {n_z:,}')
assert (n_rate_na, n_cnt_na, n_z) == (6372, 5569, 803)
miss_by_year = (df.assign(비율결측=f0['비율 결측']).groupby('공시연도')['비율결측'].mean() * 100).round(1)
print('\n공시연도별 비율 결측률(%):'); print(miss_by_year.to_string())

비율 결측 6,372 = 계 결측 5,569 + 재적 0(계 있음) 803

공시연도별 비율 결측률(%):
공시연도
2023     0.8
2024    14.8
2025    14.7
2026    15.2


In [8]:
# s3-02 [유일성] 중복
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-02)'
n_full_dup = int(df.duplicated().sum())
n_key_dup = int(df.duplicated(KEY).sum())
print('완전 중복:', n_full_dup, '| 키(공시연도·학교코드·학과코드) 중복:', n_key_dup)
assert (n_full_dup, n_key_dup) == (0, 0)

완전 중복: 0 | 키(공시연도·학교코드·학과코드) 중복: 0


In [9]:
# s3-03 [유효성] 재적 0, 계 > 재적
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-03)'
cols = ['재적 = 0', '재적 = 0 이면서 계 > 0', '재적 = 0 인데 비율 기록됨(0%)', '계 > 재적 (재적 > 0)']
print(f0[cols].sum().to_string())
assert f0['재적 = 0'].sum() == 2500 and f0['재적 = 0 이면서 계 > 0'].sum() == 1497 and f0['계 > 재적 (재적 > 0)'].sum() == 172
over = df[f0['계 > 재적 (재적 > 0)']]
print(f"\n계 > 재적 172행 중 재적 5명 이하: {(over['재적학생'] <= 5).mean() * 100:.0f}%")
# 재적 0인데 비율이 0%로 기록된 행: 계 > 0인 행도 0%로 적혀 있어 y로 쓸 수 없다 → rate_valid()에서 제외

재적 = 0                  2500
재적 = 0 이면서 계 > 0        1497
재적 = 0 인데 비율 기록됨(0%)    1697
계 > 재적 (재적 > 0)          172

계 > 재적 172행 중 재적 5명 이하: 87%


In [10]:
# s3-04 [정확성] 공시 비율 vs 계 ÷ 재적 × 100
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-04)'
v = df[rate_valid(df)]
gap = (v['중도탈락학생비율'] - v['계'] / v['재적학생'] * 100).abs()
print(f'비교 가능 행: {len(v):,}')
print(f'최대 차이: {gap.max():.4f}%p | 0.01%p 초과: {(gap > 0.01).sum()}행 | 0.005%p 초과: {(gap > 0.005 + 1e-9).sum()}행')
assert gap.max() <= 0.0051, '반올림 수준을 넘는 차이 발견 → RATE_SOURCE 결정 재검토'
# 결론: 차이는 소수 둘째 자리 반올림 수준 → 공시값 사용(RATE_SOURCE='published').
# 계획서의 '68행'은 이 정의로 재현되지 않음(정의 차이로 추정) → 정제로그에 기록

비교 가능 행: 48,271
최대 차이: 0.0050%p | 0.01%p 초과: 0행 | 0.005%p 초과: 89행


In [11]:
# s3-05 [일관성] 사유합, 공백, 표준분류, 한국외대 코드 변동
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-05)'
print('사유합 ≠ 계:', int(f0['사유합 ≠ 계'].sum()))
print('학과명 앞뒤 공백:', int((df['학과명'] != df['학과명'].str.strip()).sum()))

# 표준분류 일관성: 같은 학과명(공백 제거)이 5개교 이상에 있을 때 계열이 학교마다 갈리는가 (전체 원본 기준)
nm = df.assign(학과명s=df['학과명'].str.strip())
g = nm.groupby('학과명s').agg(학교수=('학교코드', 'nunique'), 중계열수=('표준분류_중계열', 'nunique'), 소계열수=('표준분류_소계열', 'nunique'))
g5 = g[g['학교수'] >= 5]
print(f"\n5개교 이상 학과명 {len(g5)}개 중 중계열 갈림 {(g5['중계열수'] > 1).sum()}개 ({(g5['중계열수'] > 1).mean() * 100:.0f}%), "
      f"소계열 갈림 {(g5['소계열수'] > 1).sum()}개 ({(g5['소계열수'] > 1).mean() * 100:.0f}%)")
n = len(df)
print(f"분류 불가 비율: 대계열 '광역계열' {df['표준분류_대계열'].eq('광역계열').mean() * 100:.1f}% (대계열에는 N.C.E. 값이 없음) | "
      f"중계열 N.C.E {f0['중계열 N.C.E'].mean() * 100:.1f}% | 소계열 N.C.E. {f0['소계열 N.C.E.'].mean() * 100:.1f}%")

# 국제지역학 포함 3개 소계열: 2개교 이상 학과명 중 소계열이 갈리는 것
names3 = nm.loc[nm['표준분류_소계열'].isin(B_SUBFIELDS_SENS), '학과명s'].unique()
g3 = nm[nm['학과명s'].isin(names3)].groupby('학과명s').agg(학교수=('학교코드', 'nunique'), 소계열수=('표준분류_소계열', 'nunique'))
g3 = g3[g3['학교수'] >= 2]
print(f"국제지역학 포함 3개 소계열: 2개교 이상 학과명 {len(g3)}개 중 소계열 갈림 {(g3['소계열수'] > 1).sum()}개")

# 한국외대
h = df[df['학교코드'] == HUFS_CODE]
print(f"\n한국외대 2026 공시 소계열 N.C.E.: {h[(h['공시연도'] == 2026) & (h['표준분류_소계열'] == 'N.C.E.')]['학과코드'].nunique()}개 학과")
c25 = set(h.loc[h['공시연도'] == 2025, '학과코드']); c26 = set(h.loc[h['공시연도'] == 2026, '학과코드'])
print(f'한국외대 학과코드 2025→2026: 신규 {len(c26 - c25)}, 소멸 {len(c25 - c26)}')

사유합 ≠ 계: 0
학과명 앞뒤 공백: 1

5개교 이상 학과명 508개 중 중계열 갈림 135개 (27%), 소계열 갈림 232개 (46%)
분류 불가 비율: 대계열 '광역계열' 0.4% (대계열에는 N.C.E. 값이 없음) | 중계열 N.C.E 4.8% | 소계열 N.C.E. 9.4%
국제지역학 포함 3개 소계열: 2개교 이상 학과명 44개 중 소계열 갈림 34개

한국외대 2026 공시 소계열 N.C.E.: 35개 학과
한국외대 학과코드 2025→2026: 신규 38, 소멸 24


In [12]:
# s3-06 [적시성] 폐교·신설, 패널 연결 가능성 (전체 원본 기준)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-06)'
print('학교상태:', df['학교상태'].value_counts().to_dict())
yrs = df.groupby(['학교코드', '학과코드'])['공시연도'].nunique()
print('학과 수(학교코드·학과코드):', len(yrs), '| 4개년 모두 존재:', int((yrs == 4).sum()))
ok = df[rate_valid(df)][['공시연도', '학교코드', '학과코드']]
prev = ok.assign(공시연도=ok['공시연도'] + 1)
n_pairs = len(ok.merge(prev, on=KEY))
print('전년→당해 연결 쌍 (두 해 모두 재적 > 0, 비율 유효):', n_pairs)

학교상태: {'기존': 55584, '폐교': 493, '신설': 263}
학과 수(학교코드·학과코드): 20158 | 4개년 모두 존재: 9369
전년→당해 연결 쌍 (두 해 모두 재적 > 0, 비율 유효): 29071


### ③-1 ~ ③-10 정제 단계 (정제로그 표 2와 같은 번호)

In [13]:
# s3-11 ③-1 중복 (완전 중복 → 키 중복)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-11)'
assert df.duplicated().sum() == 0 and df.duplicated(KEY).sum() == 0
log(1, '완전 중복·키 중복 확인', '0 / 0', '안내문: 완전 중복 제거를 첫 단계로', '처리 없음 (제거할 행 없음)')
track('② 중복 확인 후', df);
print('중복 0건 → 처리 없음. 행 수', len(df))

중복 0건 → 처리 없음. 행 수 56340


In [14]:
# s3-12 ③-2 공백·표기
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-12)'
df.columns = df.columns.str.strip()
str_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object]
changed = {c: int((df[c].notna() & (df[c] != df[c].str.strip())).sum()) for c in str_cols}
changed = {c: v for c, v in changed.items() if v}
for c in str_cols:
    df[c] = df[c].str.strip()
print('공백 제거한 셀:', changed)
assert len(df.columns) == 32
log(2, '컬럼명·문자열 공백 제거', sum(changed.values()), f'공백 셀 {changed}', '값 의미 변화 없음, groupby·필터 안정')

공백 제거한 셀: {'학과명': 1, '표준분류_소계열': 373}


In [15]:
# s3-13 ③-3 타입 확인
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-13)'
num_cols = ['공시연도', '기준연도', '학교코드', '학과코드', '재적학생', '계'] + REASONS + ['중도탈락학생비율']
before = df[num_cols].isna().sum().sum()
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
after = df[num_cols].isna().sum().sum()
print('숫자 컬럼 결측 수: 변환 전', before, '→ 변환 후', after)
assert before == after, '변환으로 새 결측이 생김'
log(3, '숫자 타입 확인', f'{len(num_cols)}개 컬럼', '변환 전후 결측 수 동일', '처리 없음')

숫자 컬럼 결측 수: 변환 전 56493 → 변환 후 56493


In [16]:
# s3-14 ③-4 논리 모순 표시 (삭제 금지)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-14)'
FL = make_flags(df)
for c in ['재적 = 0', '재적 = 0 이면서 계 > 0', '재적 = 0 인데 비율 기록됨(0%)', '계 > 재적 (재적 > 0)', '재계산 불일치 (> 0.01%p)', '사유합 ≠ 계']:
    df['flag_' + c] = FL[c]
print(FL[['재적 = 0', '재적 = 0 이면서 계 > 0', '재적 = 0 인데 비율 기록됨(0%)', '계 > 재적 (재적 > 0)', '재계산 불일치 (> 0.01%p)', '사유합 ≠ 계']].sum().to_string())
log(4, '재적 = 0 표시 → 표본 제외', int(FL['재적 = 0'].sum()), '비율 분모 0', '분석 표본에서 제외(행 유지)')
log(5, '재적 = 0 이면서 계 > 0 표시', int(FL['재적 = 0 이면서 계 > 0'].sum()), '분모·분자 모순. 이 중 비율이 0%로 기록된 행 포함', '재적 0 행은 모두 y로 쓰지 않음')
log(6, '계 > 재적(비율 100% 초과) 표시, 값 수정 안 함', int(FL['계 > 재적 (재적 > 0)'].sum()), '대부분 재적 5명 이하', '직전연도 최소 재적 30명으로 통제')
log(7, '공시 비율 vs 계÷재적 불일치 표시', int(FL['재계산 불일치 (> 0.01%p)'].sum()), '최대 차이 0.005%p(반올림). 계획서 68행은 재현 안 됨', '공시값 사용(팀 결정 2026-11-11)')

재적 = 0                  2500
재적 = 0 이면서 계 > 0        1497
재적 = 0 인데 비율 기록됨(0%)    1697
계 > 재적 (재적 > 0)          172
재계산 불일치 (> 0.01%p)         0
사유합 ≠ 계                    0


In [17]:
# s3-15 ③-5 결측 분해 (보정하지 않음)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-15)'
dec = pd.DataFrame({'비율 결측': FL['비율 결측'], '└ 계 결측': FL['└ 계 결측'], '└ 재적 0 (계 있음)': FL['└ 재적 0 (계 있음)']})
print(dec.groupby(df['공시연도']).sum().assign(행수=df.groupby('공시연도').size()).to_string())
assert FL['비율 결측'].sum() == FL['└ 계 결측'].sum() + FL['└ 재적 0 (계 있음)'].sum()
log(8, '비율 결측 분해 → 표본 제외', int(FL['비율 결측'].sum()), f"계 결측 {int(FL['└ 계 결측'].sum())} + 재적 0 {int(FL['└ 재적 0 (계 있음)'].sum())}", '보정 안 함(y 결측 대체 금지)')

      비율 결측  └ 계 결측  └ 재적 0 (계 있음)     행수
공시연도                                     
2023    114      46             68  14350
2024   2019    1795            224  13628
2025   2045    1770            275  13928
2026   2194    1958            236  14434


In [18]:
# s3-16 ③-6 범위 확정 (일반 대학 주간)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-16)'
conds = [
    ("학교종류 == '대학교'", df['학교종류'].eq('대학교')),
    ("주야간구분 == '주간'", df['주야간구분'].eq('주간')),
    ("학과특성 == '일반과정'", df['학과특성'].eq('일반과정')),
    ("학교상태 != '폐교'", df['학교상태'].ne('폐교')),
]
rows, keep = [], pd.Series(True, index=df.index)
for name, c in conds:
    rows.append({'조건': name, '단독 기준 제외': int((~c).sum()), '순차 적용 시 추가 제외': int((keep & ~c).sum())})
    keep &= c
scope_tbl = pd.DataFrame(rows)
df['in_scope'] = keep
print(scope_tbl.to_string(index=False))
print('\n제외 값별 건수')
for col, val in [('학교종류', '대학교'), ('주야간구분', '주간'), ('학과특성', '일반과정')]:
    print(f'  {col}:', df.loc[df[col].ne(val), col].value_counts().to_dict())
n_scope = int(df['in_scope'].sum())
by_year = df[df['in_scope']].groupby('공시연도').size().to_dict()
n_sch = df.loc[df['in_scope'], '학교코드'].nunique()
print(f'\n범위 안: {n_scope:,}행 ({n_scope / len(df) * 100:.1f}%), {n_sch}개교, 연도별 {by_year}')
assert n_scope == 50081 and by_year == {2023: 12676, 2024: 12116, 2025: 12388, 2026: 12901} and n_sch == 210
log(9, '범위 확정: 일반 대학 주간 학과(in_scope 표시)', f'{len(df):,} → {n_scope:,}', '팀 결정 2026-11-11, 가설 문서 2절', '범위 밖은 민감도 ②에만 사용(행 유지)')
track('③ 범위 안 (일반 대학 주간)', df[df['in_scope']]);

            조건  단독 기준 제외  순차 적용 시 추가 제외
 학교종류 == '대학교'      3595           3595
 주야간구분 == '주간'      3748           1626
학과특성 == '일반과정'      1249            575
  학교상태 != '폐교'       493            463

제외 값별 건수
  학교종류: {'사이버대학(대학)': 1650, '산업대학': 1091, '교육대학': 564, '방송통신대학': 148, '각종학교(대학)': 136, '기술대학': 6}
  주야간구분: {'야간': 2059, '원격': 1689}
  학과특성: {'계약학과': 879, '산업체위탁': 305, '학석사통합과정': 36, '특별과정': 29}

범위 안: 50,081행 (88.9%), 210개교, 연도별 {2023: 12676, 2024: 12116, 2025: 12388, 2026: 12901}


In [19]:
# s3-17 ③-7 표준분류 N.C.E. 표시 (재분류 안 함)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-17)'
nce = pd.DataFrame({
    '범위 안': [int((df['in_scope'] & FL['중계열 N.C.E']).sum()), int((df['in_scope'] & FL['소계열 N.C.E.']).sum())],
    '범위 밖': [int((~df['in_scope'] & FL['중계열 N.C.E']).sum()), int((~df['in_scope'] & FL['소계열 N.C.E.']).sum())],
    '한국외대(범위 안)': [int((df['in_scope'] & FL['중계열 N.C.E'] & df['학교코드'].eq(HUFS_CODE)).sum()),
                     int((df['in_scope'] & FL['소계열 N.C.E.'] & df['학교코드'].eq(HUFS_CODE)).sum())],
}, index=['중계열 N.C.E', '소계열 N.C.E.'])
print(nce)
log(10, '표준분류 N.C.E.: 재분류 안 함', f"범위 안 소계열 {nce.loc['소계열 N.C.E.', '범위 안']}, 중계열 {nce.loc['중계열 N.C.E', '범위 안']}", '학교별 신고 의존(가설 문서 3절)', "'분류 불가' 범주로 유지, 소계열은 B 대상군 정의에만 사용")

            범위 안  범위 밖  한국외대(범위 안)
중계열 N.C.E   2391   286          35
소계열 N.C.E.  4788   491          85


In [20]:
# s3-18 ③-8 패널 연결 (범위 안, 전년 → 당해)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-18)'
# 두 해 모두 범위 안인 (학교코드, 학과코드)를 t−1 → t로 잇는다. 전년도 정보는 prev_ 접두어.
FEAT = ['학교명', '학교종류', '설립구분', '설립유형', '소재지유형', '지역', '학교상태', '학과명',
        '표준분류_대계열', '표준분류_중계열', '표준분류_소계열', '재적학생', '계', '중도탈락학생비율'] + REASONS

def build_pairs(d):
    d = d.assign(rate_ok=rate_valid(d))
    cur = d[['공시연도', '학교코드', '학과코드', 'in_scope', 'rate_ok'] + FEAT]
    prv = d[['공시연도', '학교코드', '학과코드', 'in_scope', 'rate_ok'] + FEAT].copy()
    prv['공시연도'] = prv['공시연도'] + 1
    prv = prv.rename(columns={c: 'prev_' + c for c in prv.columns if c not in KEY})
    return cur.merge(prv, on=KEY, how='inner')

pairs = build_pairs(df)                         # 전체 원본 기준 쌍 (민감도 ②용 포함)
pairs['both_in_scope'] = pairs['in_scope'] & pairs['prev_in_scope']
cur_scope = df[df['in_scope'] & df['공시연도'].ge(2024)]
n_link = int(pairs['both_in_scope'].sum())
print(f'결합 전: 범위 안 t=2024~2026 행 {len(cur_scope):,} → 결합 후(전년도도 범위 안): {n_link:,}쌍, 미연결 {len(cur_scope) - n_link:,}')
hufs_cur = cur_scope[cur_scope['학교코드'] == HUFS_CODE]
hufs_link = pairs[pairs['both_in_scope'] & pairs['학교코드'].eq(HUFS_CODE)]
print('한국외대 연도별 (당해 학과 수 / 연결된 학과 수):')
print(pd.DataFrame({'당해 학과': hufs_cur.groupby('공시연도').size(), '연결': hufs_link.groupby('공시연도').size()}).fillna(0).astype(int).to_string())
log(11, '패널 연결(두 해 모두 범위 안)', f'{len(cur_scope):,} → {n_link:,}쌍', '신설·코드 변경 학과는 전년도가 없어 자연 제외', '생존 편향을 한계로 명시')
n_h_un = len(hufs_cur) - len(hufs_link)
log(12, '한국외대 학과코드 미연결', n_h_un, '학부·전공 개편', 'B 예측 절 평가에서 제외하고 건수 공개, B 판정은 군 단위 풀링(연결 불필요)')
track('④ 연결 쌍 (당해 기준)', df.loc[df['in_scope']].merge(pairs.loc[pairs['both_in_scope'], KEY], on=KEY));

결합 전: 범위 안 t=2024~2026 행 37,405 → 결합 후(전년도도 범위 안): 32,006쌍, 미연결 5,399
한국외대 연도별 (당해 학과 수 / 연결된 학과 수):
      당해 학과   연결
공시연도            
2024    129  123
2025    138  121
2026    152  114


In [21]:
# s3-19 ③-9 분석 표본 확정 (+ 민감도 표본)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-19)'
def analysis_sample(p, min_enroll=MIN_ENROLL, scope=SCOPE):
    m = p['rate_ok'] & p['prev_rate_ok'] & p['prev_재적학생'].ge(min_enroll)
    if scope == 'general_day':
        m &= p['both_in_scope']
    return p[m].copy()

steps = pd.DataFrame([
    ('연결 쌍 (두 해 모두 범위 안)', int(pairs['both_in_scope'].sum())),
    ('+ 당해 비율 유효', int((pairs['both_in_scope'] & pairs['rate_ok']).sum())),
    ('+ 전년도 비율 유효', int((pairs['both_in_scope'] & pairs['rate_ok'] & pairs['prev_rate_ok']).sum())),
    (f'+ 전년도 재적 ≥ {MIN_ENROLL}명 = 분석 표본', len(analysis_sample(pairs))),
], columns=['단계', '쌍 수'])
print(steps.to_string(index=False))

sample = analysis_sample(pairs)
sens = {f'재적≥{k}': analysis_sample(pairs, k) for k in MIN_ENROLL_SENS}
sens['전체 표본(재적≥30)'] = analysis_sample(pairs, MIN_ENROLL, 'all')
split = sample.groupby('공시연도').size()
print('\n분석 표본 공시연도별:', split.to_dict(), f'| 학습(t=2024·2025) {split.loc[TRAIN_T].sum():,} / 테스트(t=2026) {split.loc[TEST_T]:,}')
print('민감도 표본 크기:', {k: len(v) for k, v in sens.items()})
for k, v in sens.items():
    assert len(v) > 0, k
log(13, f'직전연도 재적 ≥ {MIN_ENROLL}명, 두 해 비율 유효', f'{n_link:,} → {len(sample):,}쌍', '사전 고정 설계(소규모 비율 변동 통제)', '민감도: 재적 10·50, 전체 표본')
smp_rows = df.merge(sample[KEY], on=KEY)
track('⑤ 분석 표본 (재적≥30)', smp_rows)
track('민감도 (재적≥10)', df.merge(sens['재적≥10'][KEY], on=KEY))
track('민감도 (재적≥50)', df.merge(sens['재적≥50'][KEY], on=KEY));

                    단계   쌍 수
    연결 쌍 (두 해 모두 범위 안) 32006
            + 당해 비율 유효 27316
           + 전년도 비율 유효 25867
+ 전년도 재적 ≥ 30명 = 분석 표본 23114



분석 표본 공시연도별: {2024: 7860, 2025: 7748, 2026: 7506} | 학습(t=2024·2025) 15,608 / 테스트(t=2026) 7,506
민감도 표본 크기: {'재적≥10': 25008, '재적≥30': 23114, '재적≥50': 21193, '전체 표본(재적≥30)': 25836}


In [22]:
# s3-20 ③-10 손대지 않는 것 · 최종 점검 · 추적표
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-20 = 정제로그 표 4·5)'
assert len(df) == len(df_raw), '행이 지워졌음 — 표시 후 제외 원칙 위반'
assert (df['중도탈락학생비율'].fillna(-1) == df_raw['중도탈락학생비율'].fillna(-1)).all(), 'y 값이 바뀌었음 — 보정 금지 위반'
assert sample['prev_재적학생'].min() >= MIN_ENROLL and sample['both_in_scope'].all()
assert not sample[['중도탈락학생비율', 'prev_중도탈락학생비율']].isna().any().any()
print('최종 점검 통과: 원본 행 수 유지, y 값 무변경, 표본 조건 충족\n')
track_tbl = pd.DataFrame(TRACK)
print('■ 단계별 추적표 (정제로그 표 5)'); display(track_tbl)
print('■ 정제 처리 로그 (정제로그 표 4)'); display(pd.DataFrame(LOG))

최종 점검 통과: 원본 행 수 유지, y 값 무변경, 표본 조건 충족

■ 단계별 추적표 (정제로그 표 5)


,① 원본 전체,② 중복 확인 후,③ 범위 안 (일반 대학 주간),④ 연결 쌍 (당해 기준),⑤ 분석 표본 (재적≥30),민감도 (재적≥10),민감도 (재적≥50)
(참고) 행 수,56340,56340,50081,32006,23114,25008,21193
재적 = 0,2500,2500,2261,276,0,0,0
재적 = 0 이면서 계 > 0,1497,1497,1359,221,0,0,0
재적 = 0 인데 비율 기록됨(0%),1697,1697,1516,95,0,0,0
계 > 재적 (재적 > 0),172,172,158,87,12,37,5
재계산 불일치 (> 0.01%p),0,0,0,0,0,0,0
사유합 ≠ 계,0,0,0,0,0,0,0
비율 결측,6372,6372,5585,4595,0,0,0
└ 계 결측,5569,5569,4840,4414,0,0,0
└ 재적 0 (계 있음),803,803,745,181,0,0,0


■ 정제 처리 로그 (정제로그 표 4)


,#,처리,건수,근거,영향
0,1,완전 중복·키 중복 확인,0 / 0,안내문: 완전 중복 제거를 첫 단계로,처리 없음 (제거할 행 없음)
1,2,컬럼명·문자열 공백 제거,374,"공백 셀 {'학과명': 1, '표준분류_소계열': 373}","값 의미 변화 없음, groupby·필터 안정"
2,3,숫자 타입 확인,15개 컬럼,변환 전후 결측 수 동일,처리 없음
3,4,재적 = 0 표시 → 표본 제외,2500,비율 분모 0,분석 표본에서 제외(행 유지)
4,5,재적 = 0 이면서 계 > 0 표시,1497,분모·분자 모순. 이 중 비율이 0%로 기록된 행 포함,재적 0 행은 모두 y로 쓰지 않음
5,6,"계 > 재적(비율 100% 초과) 표시, 값 수정 안 함",172,대부분 재적 5명 이하,직전연도 최소 재적 30명으로 통제
6,7,공시 비율 vs 계÷재적 불일치 표시,0,최대 차이 0.005%p(반올림). 계획서 68행은 재현 안 됨,공시값 사용(팀 결정 2026-11-11)
7,8,비율 결측 분해 → 표본 제외,6372,계 결측 5569 + 재적 0 803,보정 안 함(y 결측 대체 금지)
8,9,범위 확정: 일반 대학 주간 학과(in_scope 표시),"56,340 → 50,081","팀 결정 2026-11-11, 가설 문서 2절",범위 밖은 민감도 ②에만 사용(행 유지)
9,10,표준분류 N.C.E.: 재분류 안 함,"범위 안 소계열 4788, 중계열 2391",학교별 신고 의존(가설 문서 3절),"'분류 불가' 범주로 유지, 소계열은 B 대상군 정의에만 사용"


In [23]:
# s3-21 정제 결과 저장 (코드로만 생성, 수동 편집 금지)
# [출처 규칙] '(출처: 노트북 §③ 셀 s3-21)'
df.to_csv(f'{PROC_DIR}/rows_flagged.csv', index=False, encoding='utf-8-sig')       # 전체 행 + 표시 컬럼 + in_scope
pairs.to_csv(f'{PROC_DIR}/pairs_all.csv', index=False, encoding='utf-8-sig')        # 전년→당해 쌍 (전체, 범위 표시)
pd.DataFrame(LOG).to_csv(f'{PROC_DIR}/cleaning_log.csv', index=False, encoding='utf-8-sig')
track_tbl.to_csv(f'{PROC_DIR}/cleaning_track.csv', encoding='utf-8-sig')
for f in sorted(os.listdir(PROC_DIR)):
    print(f, os.path.getsize(f'{PROC_DIR}/{f}') // 1024, 'KB')

.gitkeep 0 KB
cleaning_log.csv 1 KB
cleaning_track.csv 0 KB
pairs_all.csv 14100 KB
rows_flagged.csv 17248 KB


## ④ 분석 (25점 — 기준선 비교 필수)
- **무엇을**: EDA·기술통계 → 분석 표본 → 모델 → 기준선 → 민감도 → 가설 A·B 판정과 각 예측 절.
- **왜**: 가설을 사전 고정된 규칙대로 판정하고, 예측 모델이 기준선보다 얼마나 나은지(또는 못한지) 정직하게 보고하려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 시간 분리(학습 t=2024·2025 / 테스트 t=2026), 누수 점검 assert, 기준선 2개와 같은 표, 민감도 표, RANDOM_STATE=42 재실행 일치

In [24]:
# s4-00 분석 공통 규칙 (주석만)
# - 무작위 분할 금지. 학습 t=2024·2025, 테스트 t=2026. RANDOM_STATE=42
# - 고위험 컷(HIGH_PCT 백분위)은 학습 데이터로만 계산해 테스트에 그대로 적용
# - 누수 금지: 당해 연도의 계·사유·비율은 설명 변수로 쓰지 않음
# - 설명 변수: 대계열, 설립유형, 소재지유형, 전년도 재적 규모(구간). 소계열은 쓰지 않음(B 대상군 정의 전용)
# - A 판정용 모델: 전년도 탈락률 제외 / 예측 성능용 모델: 전년도 탈락률 추가

### ④-1 EDA·기술통계

In [25]:
# s4-01 EDA: 재적·계 분포
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-01)'를 붙인다.
# 내용: 절대=인원(명)과 비중=비율(%)을 함께 본다. 범위 안 기준
# TODO

In [26]:
# s4-02 기술통계 (가설 A, 예측 아님)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-02)'를 붙인다.
# 내용: 설립유형 × 수도권 4칸의 고위험 비율, 시·도별 순위
# TODO

### ④-2 분석 표본과 y

In [27]:
# s4-03 y 정의
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-03)'를 붙인다.
# 내용: 학습기간 유효 학과의 HIGH_PCT 백분위 컷 → 테스트에 그대로 적용. 컷 값과 학습/테스트 양성 비율 출력
# TODO

In [28]:
# s4-04 설명 변수와 누수 점검
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-04)'를 붙인다.
# 내용: 변수 목록 출력, 당해 연도 변수 미포함 assert
# TODO

### ④-3 가설 A: 전국 구조

In [29]:
# s4-05 A 판정용 로지스틱 회귀 (statsmodels)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-05)'를 붙인다.
# 내용: 사립·비수도권 + 대계열 + 전년도 재적 규모. 오즈비와 95% CI 표 → 판정 규칙(맞음/부분/기각) 적용
# TODO

### ④-4 A 안의 예측 절

In [30]:
# s4-06 모델 2개 학습
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-06)'를 붙인다.
# 내용: 구조 모델 / 구조 + 전년도 탈락률 모델 (학습 t=2024·2025)
# TODO

In [31]:
# s4-07 기준선 2개
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-07)'를 붙인다.
# 내용: ① 전원 '고위험 아님' ② 전년도 고위험 지속
# TODO

In [32]:
# s4-08 테스트 평가표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-08)'를 붙인다.
# 내용: k = 기준선 ②가 고른 학과 수. 같은 k의 정밀도·재현율(주, 맞힌 학과 수 함께), 보조로 정확도·AUC·혼동행렬을 기준선과 한 표로. 임계값 0.5는 민감도 ⑤에만. 판정 임계값 없음
# TODO

### ④-5 가설 B: 한국외대

In [33]:
# s4-09 B 대상군 정의
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-09)'를 붙인다.
# 내용: 한국외대 + 소계열 in B_SUBFIELDS. 연도별 학과 수·재적 합계
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 매년 23개 학과
# TODO

In [34]:
# s4-10 B 판정
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-10)'를 붙인다.
# 내용: 2024~2026 합산 비율(탈락 합계 ÷ 재적 합계, %) 대상군 vs 교내 다른 학과, 학과코드 단위 부트스트랩 95% 구간(RANDOM_STATE 고정) → 판정
# TODO

In [35]:
# s4-11 B 보조 분석
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-11)'를 붙인다.
# 내용: 대상군 안에서 전년도 재적 하위 25%(소규모)가 더 높은지 (재적 대리 지표)
# TODO

In [36]:
# s4-12 B 참고 해석 (가설 아님)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-12)'를 붙인다.
# 내용: 전국(한국외대 제외) 같은 소계열과 비교만. 미리 고정한 결과 문구 중 하나를 선택
# 기대값(assert, 계획서 기준값 — 재실행으로 확정): 비교군 8~9개교, 31~43개 학과
# TODO

### ④-6 B 안의 예측 절 (보조 참고)

In [37]:
# s4-13 한국외대 예측
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-13)'를 붙인다.
# 내용: A 모델로 2026 공시 한국외대 학과의 다음 해 고위험 점수 → 우선 지원 후보 상위 k개. t=2026 테스트의 한국외대 학과(약 130개)에서 기준선 ②와 정밀도 비교. 미연결 학과 건수 공개
# TODO

### ④-7 민감도 (사전 고정 ①~⑤)

In [38]:
# s4-14 민감도 표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-14)'를 붙인다.
# 내용: ① B에 국제지역학 소계열 추가 ② 전체 표본으로 A 재추정 ③ 최소 재적 10·30·50 ④ 고위험 컷 상위 20%·30% ⑤ k 기준 vs 임계값 0.5(비용 가정 명시). 결론이 뒤집히는지 한 표로
# TODO

### ④-8 판정 요약

In [39]:
# s4-15 가설 판정 표
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §④ 셀 s4-15)'를 붙인다.
# 내용: 가설 | 판정(맞음/부분/기각) | 근거 수치 | 셀 ID | 판정에 따른 행동
# TODO

## ⑤ 발표 (시각화·권고·근거·한계)
- **무엇을**: 필수 차트 3종을 그리고, 권고·근거·한계와 '믿기 어려운 이유'를 정리한다.
- **왜**: 차트마다 '그래서 무엇' 한 줄과 출처를 붙여 슬라이드(`slides/발표_구성.md`)로 바로 옮기려고.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: 그림 파일(figures/), 축 단위(명/%), 그림 아래 출처 줄, 슬라이드 구성표의 ⚠ 0개

### ⑤-1 후보 차트 갤러리 (프로젝트1 방식: 가능한 차트를 먼저 다 그려 보고 베스트 3종을 고른다)
> 각 셀 제목에 `[단독/서브플롯, 베스트 후보]`를 단다. 모두 `figures/cand_XX_*.png`로 저장한다.
> **선택 기준**(⑤-2 표에 점수로 기록): ① 의사결정(어느 학과에 지원할까)에 직접 답하는가 ② 가설 A·B·예측 절과 연결되는가 ③ '그래서 무엇' 한 줄이 바로 나오는가 ④ 안내문 요건(교차는 산점도, 장식 차트 금지) ⑤ 10분 안에 설명 가능한가
> 필수 구성: 교차 산점도 1개 + 가설 B 비교 1개 + 모델 vs 기준선 1개. 탈락 차트는 부록 D 후보로 남긴다.

In [40]:
# s5-c01 [단독, 베스트 후보·필수 교차] 탈락 인원(명) × 탈락률(%) 산점도 — 범위 안 학과, 재적 크기 색, 최소 재적선, 한국외대 강조
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c01)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [41]:
# s5-c02 [단독] 재적 규모 구간별 탈락률 상자그림 — 소규모 학과의 비율 변동(최소 재적 30명 근거)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c02)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [42]:
# s5-c03 [서브플롯] 공시연도별 비율 결측률·재적 0 건수 막대 — 함정(공시 정의·결측 패턴 변화)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c03)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [43]:
# s5-c04 [단독, 베스트 후보] 정제 추적 퍼널: 원본 → 범위 → 연결 쌍 → 분석 표본 (행 수)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c04)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [44]:
# s5-c05 [단독, 베스트 후보] 전년도 탈락률 × 당해 탈락률 산점도 — 지속성(기준선 ②의 근거)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c05)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [45]:
# s5-c06 [단독] 설립유형 × 수도권 4칸 고위험 비율 히트맵 (가설 A 기술통계)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c06)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [46]:
# s5-c07 [단독] 시·도별 고위험 비율 막대 (가설 A 기술통계, 학과 수 함께 표기)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c07)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [47]:
# s5-c08 [단독, 베스트 후보] 가설 A 오즈비 포레스트 플롯 (95% CI, 기준선 1)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c08)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [48]:
# s5-c09 [단독, 베스트 후보·필수 B] 한국외대 학과군별 2024~2026 합산 탈락률 막대 + 부트스트랩 구간
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c09)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [49]:
# s5-c10 [단독] 한국외대 학과별 인원(명) × 비율(%) 산점도 — 비주류 외국어 학과군 강조
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c10)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [50]:
# s5-c11 [단독] 한국외대 vs 전국 같은 소계열 비교 도트 플롯 (B 참고 해석, 구간 포함)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c11)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [51]:
# s5-c12 [단독, 베스트 후보·필수 예측] 모델 2개 vs 기준선 2개 정밀도 막대 (같은 k)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c12)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [52]:
# s5-c13 [단독, 베스트 후보] k 변화에 따른 정밀도 곡선 — 모델 vs 기준선 ② (지원 가능 학과 수 선택 근거)
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c13)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

In [53]:
# s5-c14 [단독] 민감도 요약 도트 플롯 — 조건별 판정·정밀도가 뒤집히는지
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §⑤ 셀 s5-c14)'를 붙인다.
# TODO: 그림 저장 → figures/cand_*.png, 축 단위(명/%), 그림 아래 출처 문구, '그래서 무엇' 한 줄 출력

### ⑤-2 베스트 3종 선택 (후보를 다 그린 뒤 작성)
| 후보 | ① 의사결정 | ② 가설 연결 | ③ 그래서 무엇 | ④ 요건 | ⑤ 설명 시간 | 합계 | 선택 | 선택·탈락 이유 |
|---|---|---|---|---|---|---|---|---|
| s5-c01 | | | | | | | | |
| … | | | | | | | | |

선택한 3종은 `figures/fig1_scatter.png`, `fig2_hufs_groups.png`, `fig3_model_vs_baseline.png`로 다시 저장한다(보고서 `make_report.js`가 이 이름을 읽음). 선택 이유는 `slides/발표_구성.md` 차트 표에 옮긴다.

### 권고·근거 (결과 확인 후 작성)
| 가설 | 판정 | 근거 수치 (셀 ID) | 권고 |
|---|---|---|---|
| A | | | |
| B | | | |

### 한계 (빠뜨리면 감점)
1. 테스트는 공시 2026 한 해뿐이다.
2. 학과 단위 집계라 개인 이탈 요인을 담지 못하며, 결과는 상관이지 인과가 아니다(A의 오즈비는 연관).
3. 폐지·통폐합·신설 학과가 연결에서 빠지는 생존 편향, 공시 정의·결측 패턴 변화. 한국외대 학과코드 변동(B는 군 단위 풀링).
- 추가: 표준분류는 학교별 신고에 의존해 같은 이름의 학과가 학교마다 다르게 분류될 수 있다. 우리는 학과를 재분류하지 않고 데이터가 준 분류를 그대로 쓴다. / "비인기"·"존폐 위기"는 판단 불가 / 캠퍼스 미구분 / 전국 비교군 소표본 / 학부·전공 이중 계상 가능성 `[확인 필요]`

### 믿기 어려운 이유 · 말할 수 없는 것 (결과 확인 후 작성)
- `[팀 입력]`

## 부록 A. 재현성 확인
- **무엇을**: 런타임 재시작 후 처음부터 모두 실행해 핵심 수치를 한 셀에 모아 출력한다.
- **왜**: 다른 계정에서 돌려도 같은 숫자가 나와야 한다(안내문 마지막 점검).
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: `logs/재현성_확인표.md`의 항목과 같은 이름으로 출력, 값이 다르면 '발견'으로 기록

In [54]:
# sA-01 재현성 요약
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §부록 A 셀 sA-01)'를 붙인다.
# TODO: 원본 행×열, 범위 안 행 수, 분석 표본(학습/테스트), A 오즈비, B 차이·구간, 같은 k 정밀도, AUC, 기준선 ②, 실행 일시, 라이브러리 버전

## 부록 E. 보조 분석 R1 (인원×비율 착시, 가설 판정 아님)
- **무엇을**: 한국외대에서 탈락 비율(%) 상위 10 학과와 탈락 인원(명) 상위 10 학과의 교집합 수를 센다(공시 2024~2026 합산).
- **왜**: 프로젝트1의 절대×비중 교훈: 비율만 보고 지원하면 큰 학과의 이탈 인원을 놓칠 수 있다.
- **초안 담당자**: `[팀 입력]` · **검토자**: `[팀 입력]` (`logs/기여도표.md`와 일치)
- **이 단계의 검증 흔적**: `00_문제정의_가설.md` 10절 규칙(교집합 < 5) 그대로, 두 목록과 교집합 출력

In [55]:
# sE-01 R1 두 상위 10 목록과 교집합
# [출처 규칙] 이 셀의 수치를 보고서·발표에 옮길 때 '(출처: 노트북 §부록 E 셀 sE-01)'를 붙인다.
# TODO: 학과(학과코드)별 2024~2026 합산 탈락 인원(명)·합산 비율(%) → 각 상위 10 → 교집합 수, 규칙 문구 출력